In [2]:
import rasterio
from rasterio.warp import transform
import numpy as np
from scipy.interpolate import griddata
import xarray as xr

In [3]:
def get_weather_for_satellite_extent(satellite_tif_path, weather_ds, date, 
                                      vars_to_use=['tmax', 'tmin', 'prcp', 'srad']):
    """
    Satellite 이미지가 커버하는 지역의 weather만 추출
    
    Returns:
        (224, 224, num_vars) - satellite와 정확히 같은 지역
    """
    # 1. Satellite의 각 픽셀 좌표 (lat/lon) 구하기
    with rasterio.open(satellite_tif_path) as src:
        rows, cols = np.meshgrid(np.arange(224), np.arange(224), indexing='ij')
        xs, ys = rasterio.transform.xy(src.transform, rows.flatten(), cols.flatten())
        xs = np.array(xs).reshape(224, 224)
        ys = np.array(ys).reshape(224, 224)
        
        # 투영좌표 -> lat/lon 변환
        lons, lats = transform(src.crs, 'EPSG:4326', xs.flatten(), ys.flatten())
        sat_lats = np.array(lats).reshape(224, 224)
        sat_lons = np.array(lons).reshape(224, 224)
    
    # 2. 해당 날짜의 weather 데이터
    weather_day = weather_ds.sel(time=date)
    
    # Weather의 lat/lon
    weather_lat = weather_day['lat'].values.flatten()
    weather_lon = weather_day['lon'].values.flatten()
    
    # Satellite 픽셀 좌표들
    sat_points = np.column_stack([sat_lons.flatten(), sat_lats.flatten()])
    
    # 3. 각 변수마다 Satellite 좌표에 맞춰 interpolation
    features = []
    for var in vars_to_use:
        weather_var = weather_day[var].values.flatten()
        
        # NaN 제거
        valid_mask = ~np.isnan(weather_var)
        weather_points = np.column_stack([weather_lon[valid_mask], 
                                          weather_lat[valid_mask]])
        weather_values = weather_var[valid_mask]
        
        # Satellite 픽셀 위치에서의 weather 값 interpolation
        matched_1d = griddata(
            weather_points,   # weather 격자점 좌표
            weather_values,   # weather 값
            sat_points,       # satellite 픽셀 좌표 (224x224개)
            method='nearest'  # 또는 'linear'
        )
        
        matched_2d = matched_1d.reshape(224, 224)
        matched_2d = np.nan_to_num(matched_2d, nan=0.0)
        features.append(matched_2d)
    
    # (224, 224, num_vars)
    return np.stack(features, axis=-1)



In [6]:
weather_ds = xr.open_dataset('/work/mech-ai-scratch/rtali/gis-weather/final_processed_weather/daymet_iowa_2023.nc')
satellite_path = '/work/mech-ai-scratch/bgekim/project/imputation/IA_dataset/30m/Patches/Elevation/patch_9968_9968/elevation_IA.tif'

weather_aligned = get_weather_for_satellite_extent(
    satellite_path,
    weather_ds,
    '2023-09-04',
    vars_to_use=['tmax', 'tmin', 'prcp', 'srad']
)

# print(f"Shape: {weather_aligned.shape}")  # (224, 224, 4)
# print(f"Satellite와 정확히 같은 6.7km x 6.7km 영역의 weather")

Processing satellite: /work/mech-ai-scratch/bgekim/project/imputation/IA_dataset/30m/Patches/Elevation/patch_9968_9968/elevation_IA.tif
Processing date: 2023-09-04
Variables: ['tmax', 'tmin', 'prcp', 'srad']

[1/3] Reading satellite coordinates...
  - Satellite CRS: EPSG:32615
  - Satellite bounds: BoundingBox(left=501113.80870045704, bottom=4516913.786513407, right=507833.80870045704, top=4523633.786513407)
  - Projected X range: 501128.81 ~ 507818.81
  - Projected Y range: 4516928.79 ~ 4523618.79
  - Satellite Lat range: 40.803325 ~ 40.863628
  - Satellite Lon range: -92.986618 ~ -92.907222

[2/3] Loading weather data for 2023-09-04...
  - Weather Lat range: nan ~ nan
  - Weather Lon range: nan ~ nan
  - Weather grid points: 199969
  - Satellite pixels to match: 50176

[3/3] Interpolating weather variables...
  - Processing tmax (1/4)...
    Valid points: 170082, NaN points: 29887
    tmax range: 32.09 ~ 36.89
    Matched tmax range: 33.96 ~ 34.42
    Matched tmax mean: 34.20, std: 0

In [5]:
import rasterio
from rasterio.warp import transform
import numpy as np
from scipy.interpolate import griddata
import xarray as xr

def get_weather_for_satellite_extent(satellite_tif_path, weather_ds, date, 
                                      vars_to_use=['tmax', 'tmin', 'prcp', 'srad']):
    """
    Satellite 이미지가 커버하는 지역의 weather만 추출
    
    Returns:
        (224, 224, num_vars) - satellite와 정확히 같은 지역
    """
    print(f"Processing satellite: {satellite_tif_path}")
    print(f"Processing date: {date}")
    print(f"Variables: {vars_to_use}")
    
    # 1. Satellite의 각 픽셀 좌표 (lat/lon) 구하기
    print("\n[1/3] Reading satellite coordinates...")
    with rasterio.open(satellite_tif_path) as src:
        print(f"  - Satellite CRS: {src.crs}")
        print(f"  - Satellite bounds: {src.bounds}")
        
        rows, cols = np.meshgrid(np.arange(224), np.arange(224), indexing='ij')
        xs, ys = rasterio.transform.xy(src.transform, rows.flatten(), cols.flatten())
        xs = np.array(xs).reshape(224, 224)
        ys = np.array(ys).reshape(224, 224)
        
        print(f"  - Projected X range: {xs.min():.2f} ~ {xs.max():.2f}")
        print(f"  - Projected Y range: {ys.min():.2f} ~ {ys.max():.2f}")
        
        # 투영좌표 -> lat/lon 변환
        lons, lats = transform(src.crs, 'EPSG:4326', xs.flatten(), ys.flatten())
        sat_lats = np.array(lats).reshape(224, 224)
        sat_lons = np.array(lons).reshape(224, 224)
        
        print(f"  - Satellite Lat range: {sat_lats.min():.6f} ~ {sat_lats.max():.6f}")
        print(f"  - Satellite Lon range: {sat_lons.min():.6f} ~ {sat_lons.max():.6f}")
    
    # 2. 해당 날짜의 weather 데이터
    print(f"\n[2/3] Loading weather data for {date}...")
    weather_day = weather_ds.sel(time=date)
    
    # Weather의 lat/lon
    weather_lat = weather_day['lat'].values.flatten()
    weather_lon = weather_day['lon'].values.flatten()
    
    print(f"  - Weather Lat range: {weather_lat.min():.6f} ~ {weather_lat.max():.6f}")
    print(f"  - Weather Lon range: {weather_lon.min():.6f} ~ {weather_lon.max():.6f}")
    print(f"  - Weather grid points: {len(weather_lat)}")
    
    # Satellite 픽셀 좌표들
    sat_points = np.column_stack([sat_lons.flatten(), sat_lats.flatten()])
    print(f"  - Satellite pixels to match: {len(sat_points)}")
    
    # 3. 각 변수마다 Satellite 좌표에 맞춰 interpolation
    print(f"\n[3/3] Interpolating weather variables...")
    features = []
    for idx, var in enumerate(vars_to_use):
        print(f"  - Processing {var} ({idx+1}/{len(vars_to_use)})...")
        weather_var = weather_day[var].values.flatten()
        
        # NaN 제거
        valid_mask = ~np.isnan(weather_var)
        num_valid = valid_mask.sum()
        num_nan = (~valid_mask).sum()
        print(f"    Valid points: {num_valid}, NaN points: {num_nan}")
        
        weather_points = np.column_stack([weather_lon[valid_mask], 
                                          weather_lat[valid_mask]])
        weather_values = weather_var[valid_mask]
        
        print(f"    {var} range: {weather_values.min():.2f} ~ {weather_values.max():.2f}")
        
        # Satellite 픽셀 위치에서의 weather 값 interpolation
        matched_1d = griddata(
            weather_points,   # weather 격자점 좌표
            weather_values,   # weather 값
            sat_points,       # satellite 픽셀 좌표 (224x224개)
            method='nearest'  # 또는 'linear'
        )
        
        matched_2d = matched_1d.reshape(224, 224)
        matched_2d = np.nan_to_num(matched_2d, nan=0.0)
        
        print(f"    Matched {var} range: {matched_2d.min():.2f} ~ {matched_2d.max():.2f}")
        print(f"    Matched {var} mean: {matched_2d.mean():.2f}, std: {matched_2d.std():.2f}")
        
        features.append(matched_2d)
    
    result = np.stack(features, axis=-1)
    print(f"\n✓ Complete! Output shape: {result.shape}")
    
    return result